In [ ]:
# @title
# Gerekli kütüphaneler
!pip install -q imageio gymnasium stable-baselines3

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import random
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import imageio
from tqdm import tqdm

# ============================================
# 10x10 GRID, 18 KONTEYNER, 4x4 İSTİF ALANI, İKİ DQN AJANI
# ============================================
class ContainerYardEnv(gym.Env):
    def __init__(self):
        super().__init__()
        self.rows = 10
        self.cols = 10
        self.max_steps = 2000

        self.action_space = spaces.Discrete(6)  # her iki ajan için de 6 eylem
        # State boyutu hesaplama:
        # crane1(2) + crane2(2) + picked(18) + stacking(4x4=16) + orders(18) + carrying1(1) + carrying2(1) + target(3) + next_pickup(2) + priority(1)
        self.state_size = 2 + 2 + 18 + 16 + 18 + 1 + 1 + 3 + 2 + 1  # = 64
        self.observation_space = spaces.Box(low=0, high=1, shape=(self.state_size,), dtype=np.float32)

        # 18 konteyner (eski liste)
        raw_containers = {
            0: {"pickup": (0,4), "shape": (1,2), "priority": 0},
            1: {"pickup": (2,4), "shape": (1,3), "priority": 0},
            2: {"pickup": (4,4), "shape": (3,1), "priority": 0},
            3: {"pickup": (7,4), "shape": (2,1), "priority": 0},
            4: {"pickup": (0,8), "shape": (1,2), "priority": 0},
            5: {"pickup": (3,8), "shape": (2,1), "priority": 0},
            6: {"pickup": (1,7), "shape": (1,2), "priority": 1},
            7: {"pickup": (5,7), "shape": (1,3), "priority": 1},
            8: {"pickup": (2,9), "shape": (2,1), "priority": 1},
            9: {"pickup": (6,7), "shape": (3,1), "priority": 1},
            10: {"pickup": (9,7), "shape": (1,2), "priority": 1},
            11: {"pickup": (4,6), "shape": (3,1), "priority": 1},
            12: {"pickup": (1,4), "shape": (1,2), "priority": 2},
            13: {"pickup": (3,5), "shape": (1,3), "priority": 2},
            14: {"pickup": (5,5), "shape": (2,1), "priority": 2},
            15: {"pickup": (8,5), "shape": (2,1), "priority": 2},
            16: {"pickup": (8,8), "shape": (1,2), "priority": 2},
            17: {"pickup": (0,6), "shape": (2,1), "priority": 2}
        }

        # containers_original'a dönüştür
        self.containers_original = {}
        for cid, data in raw_containers.items():
            r, c = data['pickup']
            rows, cols = data['shape']
            cells = [(r + i, c + j) for i in range(rows) for j in range(cols)]
            priority = data['priority']
            color = 'green' if priority == 0 else ('yellow' if priority == 1 else 'red')
            self.containers_original[cid] = {
                'cells': cells,
                'size': (rows, cols),
                'name': f'K{cid}',
                'pickup': data['pickup'],
                'priority': priority,
                'color': color
            }

        # Bloklu alanlar (satır 0-2 ve 7-9, sütun 0-3)
        self.blocked_cells = []
        for r in range(3):
            for c in range(4):
                self.blocked_cells.append((r, c))
        for r in range(7, 10):
            for c in range(4):
                self.blocked_cells.append((r, c))

        # İstif alanı: satır 3-6, sütun 0-3 (4x4)
        self.stacking_height = 4
        self.stacking_width = 4

        self.crane2_home = (9, 7)  # Vinç-2 bekleme noktası (sağ alt)
        self.reset()

    def reset(self, seed=None):
        super().reset(seed=seed)
        # Rastgele 12 sipariş
        all_ids = list(self.containers_original.keys())
        self.orders = random.sample(all_ids, 12)

        self.crane1_pos = [3, 0]   # Vinç-1 istif alanı içinde başlar
        self.crane2_pos = [9, 7]   # Vinç-2 bekleme noktası
        self.relocated = []        # Vinç-2 tarafından taşınanlar (cid, original_cells, original_pickup)

        self.containers = {}
        for cid, data in self.containers_original.items():
            self.containers[cid] = {
                'cells': data['cells'].copy(),
                'original_cells': data['cells'].copy(),
                'size': data['size'],
                'name': data['name'],
                'pickup': data['pickup'],
                'original_pickup': data['pickup'],
                'priority': data['priority'],
                'color': data['color'],
                'picked': False,
                'delivered': False,
                'rotated': False,
                'relocated': False
            }

        self.stacking_grid = [[[] for _ in range(self.stacking_width)] for _ in range(self.stacking_height)]
        self.steps = 0
        self.carrying1 = None
        self.carrying2 = None
        self.done = False
        self.rotation_msg = ""
        self.target_pos = None
        self.next_pickup_target = None
        self.expected_priority = None

        self._update_targets()
        self._prev_distance1 = self._get_distance(self.crane1_pos, self._get_current_target())
        return self._get_obs(), {}

    # ---------- Yardımcı Fonksiyonlar ----------
    def _get_distance(self, pos, target):
        if target is None: return 0.0
        return abs(pos[0] - target[0]) + abs(pos[1] - target[1])

    def _get_current_target(self):
        return self.target_pos if self.carrying1 is not None else self.next_pickup_target

    def _get_obs(self):
        obs = np.zeros(self.state_size, dtype=np.float32)
        obs[0] = (self.crane1_pos[0] - 3) / 3.0
        obs[1] = self.crane1_pos[1] / (self.cols - 1)
        obs[2] = (self.crane2_pos[0] - 3) / 3.0
        obs[3] = self.crane2_pos[1] / (self.cols - 1)

        # picked (18 adet)
        for i in range(18):
            obs[4 + i] = float(self.containers[i]['picked'])

        # stacking grid (16 hücre)
        idx = 4 + 18
        for r in range(self.stacking_height):
            for c in range(self.stacking_width):
                stack = self.stacking_grid[r][c]
                obs[idx] = stack[-1] / 17.0 if stack else 0.0
                idx += 1

        # orders (18 adet)
        for i in range(18):
            obs[idx + i] = 1.0 if i in self.orders else 0.0
        idx += 18

        obs[idx] = float(self.carrying1 is not None)
        idx += 1
        obs[idx] = float(self.carrying2 is not None)
        idx += 1

        if self.target_pos is not None:
            obs[idx] = (self.target_pos[0] - 3) / 3.0
            obs[idx+1] = self.target_pos[1] / 3.0
            obs[idx+2] = self.target_pos[2] / 4.0
        idx += 3

        if self.next_pickup_target is not None:
            obs[idx] = self.next_pickup_target[0] / (self.rows - 1)
            obs[idx+1] = self.next_pickup_target[1] / (self.cols - 1)
        idx += 2

        if self.expected_priority is not None:
            obs[idx] = self.expected_priority / 2.0

        return obs

    def _is_blocked(self, pos, crane_id):
        r, c = pos
        if r < 0 or r >= self.rows or c < 0 or c >= self.cols:
            return True
        if (r, c) in self.blocked_cells:
            return True
        # Diğer vinç her zaman engel
        other = self.crane2_pos if crane_id == 1 else self.crane1_pos
        if (r, c) == tuple(other):
            return True

        # Vinç-1 sadece yük taşırken konteynerler engel
        if crane_id == 1 and self.carrying1 is not None:
            for cid, data in self.containers.items():
                if not data['picked'] and (r, c) in data['cells']:
                    return True

        # Vinç-2 için konteynerler ASLA engel değil
        if crane_id == 2:
            return False

        return False

    def _is_in_stacking_area(self, pos):
        r, c = pos
        return 3 <= r <= 6 and 0 <= c <= 3

    def _is_in_temp_storage(self, pos):
        """Geçici depolama alanı: sütun 7-9"""
        r, c = pos
        return 0 <= r < self.rows and 7 <= c <= 9

    def _get_container_at(self, pos):
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered'] and pos == data['pickup']:
                return cid
        return None

    def _find_empty_temp_spot(self, size):
        rows, cols = size
        for r in range(self.rows - rows + 1):
            for c in range(7, self.cols - cols + 1):
                if (r, c) == self.crane2_home: continue
                valid = True
                for i in range(rows):
                    for j in range(cols):
                        nr, nc = r+i, c+j
                        if (nr, nc) == self.crane2_home or self._is_in_stacking_area((nr, nc)) or (nr, nc) in self.blocked_cells:
                            valid = False; break
                        for cid, data in self.containers.items():
                            if not data['picked'] and (nr, nc) in data['cells']:
                                valid = False; break
                        if not valid: break
                    if not valid: break
                if valid:
                    return (r, c)
        return None

    def _find_stacking_position(self, container_id, rotated=False):
        size = self.containers[container_id]['size']
        prio = self.containers[container_id]['priority']
        if rotated: size = (size[1], size[0])
        rows, cols = size
        if rows > self.stacking_height or cols > self.stacking_width: return None
        for level in range(5):
            for r in range(self.stacking_height - rows + 1):
                for c in range(self.stacking_width - cols + 1):
                    ok = True
                    for i in range(rows):
                        for j in range(cols):
                            stack = self.stacking_grid[r+i][c+j]
                            if len(stack) > level: ok = False; break
                            if level > 0:
                                if len(stack) <= level-1: ok = False; break
                                below = stack[level-1]
                                if prio <= self.containers[below]['priority']: ok = False; break
                        if not ok: break
                    if ok: return (3+r, c, level), size
        return None

    def _update_targets(self):
        if self.carrying1 is not None:
            pos_info = self._find_stacking_position(self.carrying1, rotated=False)
            rotated = False
            if pos_info is None:
                pos_info = self._find_stacking_position(self.carrying1, rotated=True)
                rotated = True
            self.target_pos = pos_info[0] if pos_info else None
            self.rotated_needed = rotated
        else:
            self.target_pos = None
            self.rotated_needed = False

        remaining = [oid for oid in self.orders if not self.containers[oid]['picked'] and not self.containers[oid]['delivered']]
        if remaining:
            remaining.sort(key=lambda cid: (self.containers[cid]['priority'], self._get_distance(self.crane1_pos, self.containers[cid]['pickup'])))
            nxt = remaining[0]
            self.next_pickup_target = self.containers[nxt]['pickup']
            self.expected_priority = self.containers[nxt]['priority']
        else:
            self.next_pickup_target = None
            self.expected_priority = None

    # ---------- Adım (step) - İki Ajan İçin ----------
    def step(self, action1, action2):
        self.steps += 1
        reward = -0.1
        self.rotation_msg = ""

        # ========== Vinç-2 hareketi (DQN) ==========
        if action2 == 0:   # Up
            new_pos = [self.crane2_pos[0]-1, self.crane2_pos[1]]
            if not self._is_blocked(new_pos, crane_id=2): self.crane2_pos = new_pos
            else: reward -= 0.5
        elif action2 == 1: # Down
            new_pos = [self.crane2_pos[0]+1, self.crane2_pos[1]]
            if not self._is_blocked(new_pos, crane_id=2): self.crane2_pos = new_pos
            else: reward -= 0.5
        elif action2 == 2: # Left
            new_pos = [self.crane2_pos[0], self.crane2_pos[1]-1]
            if not self._is_blocked(new_pos, crane_id=2): self.crane2_pos = new_pos
            else: reward -= 0.5
        elif action2 == 3: # Right
            new_pos = [self.crane2_pos[0], self.crane2_pos[1]+1]
            if not self._is_blocked(new_pos, crane_id=2): self.crane2_pos = new_pos
            else: reward -= 0.5
        elif action2 == 4: # Pickup
            if self.carrying2 is None:
                cid = self._get_container_at(tuple(self.crane2_pos))
                if cid is not None and cid not in self.orders and not self.containers[cid]['relocated']:
                    self.containers[cid]['picked'] = True
                    self.carrying2 = cid
                    spot = self._find_empty_temp_spot(self.containers[cid]['size'])
                    if spot:
                        self.crane2_temp_pos = spot
                    else:
                        self.containers[cid]['picked'] = False
                        self.carrying2 = None
                        reward -= 2.0
                else:
                    reward -= 2.0
            else:
                reward -= 2.0
        elif action2 == 5: # Drop
            if self.carrying2 is not None:
                if self.crane2_temp_pos is not None and tuple(self.crane2_pos) == self.crane2_temp_pos:
                    cid = self.carrying2
                    r, c = self.crane2_temp_pos
                    rows, cols = self.containers[cid]['size']
                    new_cells = [(r+i, c+j) for i in range(rows) for j in range(cols)]
                    self.containers[cid]['cells'] = new_cells
                    self.containers[cid]['pickup'] = new_cells[0]
                    self.containers[cid]['picked'] = False
                    self.containers[cid]['relocated'] = True
                    self.relocated.append((cid, self.containers[cid]['original_cells'].copy(), self.containers[cid]['original_pickup']))
                    self.carrying2 = None
                    self.crane2_temp_pos = None
                    reward += 5.0
                elif self._is_in_temp_storage(tuple(self.crane2_pos)):
                    cid = self.carrying2
                    spot = self._find_empty_temp_spot(self.containers[cid]['size'])
                    if spot is not None and tuple(self.crane2_pos) == spot:
                        r, c = spot
                        rows, cols = self.containers[cid]['size']
                        new_cells = [(r+i, c+j) for i in range(rows) for j in range(cols)]
                        self.containers[cid]['cells'] = new_cells
                        self.containers[cid]['pickup'] = new_cells[0]
                        self.containers[cid]['picked'] = False
                        self.containers[cid]['relocated'] = True
                        self.relocated.append((cid, self.containers[cid]['original_cells'].copy(), self.containers[cid]['original_pickup']))
                        self.carrying2 = None
                        reward += 5.0
                    else:
                        reward -= 2.0
                else:
                    reward -= 2.0
            else:
                reward -= 2.0

        # ========== Vinç-1 hareketi (DQN) ==========
        current_target = self._get_current_target()
        if current_target is not None:
            old_dist = self._prev_distance1
            new_dist = self._get_distance(self.crane1_pos, current_target)
            if new_dist < old_dist:
                reward += 0.3
            elif new_dist > old_dist:
                reward -= 0.3
            self._prev_distance1 = new_dist

        if self.carrying1 is None and self.next_pickup_target is not None:
            if tuple(self.crane1_pos) == self.next_pickup_target and action1 != 4:
                reward -= 0.5

        blocked = False
        if action1 == 0:   # Up
            new_pos = [self.crane1_pos[0]-1, self.crane1_pos[1]]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action1 == 1: # Down
            new_pos = [self.crane1_pos[0]+1, self.crane1_pos[1]]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action1 == 2: # Left
            new_pos = [self.crane1_pos[0], self.crane1_pos[1]-1]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action1 == 3: # Right
            new_pos = [self.crane1_pos[0], self.crane1_pos[1]+1]
            if not self._is_blocked(new_pos, crane_id=1): self.crane1_pos = new_pos
            else: blocked = True
        elif action1 == 4: # Pickup
            if self.carrying1 is None:
                cid = self._get_container_at(tuple(self.crane1_pos))
                if cid is not None and cid in self.orders:
                    if self.containers[cid]['priority'] == self.expected_priority:
                        self.containers[cid]['picked'] = True
                        self.carrying1 = cid
                        self._update_targets()
                        reward += 8.0
                        self._prev_distance1 = self._get_distance(self.crane1_pos, self._get_current_target())
                    else: reward -= 5.0
                else: reward -= 2.0
            else: reward -= 2.0
        elif action1 == 5: # Drop
            if self.carrying1 is not None:
                if not self._is_in_stacking_area(tuple(self.crane1_pos)):
                    reward -= 4.0; self.rotation_msg = "Sadece istif alaninda birakabilirsiniz!"
                elif self.target_pos is None:
                    reward -= 4.0; self.rotation_msg = "Yer yok!"
                elif (self.crane1_pos[0], self.crane1_pos[1]) != (self.target_pos[0], self.target_pos[1]):
                    reward -= 4.0; self.rotation_msg = f"Hedef: {self.target_pos[:2]}"
                else:
                    reward += 2.0
                    cid = self.carrying1
                    pos_info = self._find_stacking_position(cid, rotated=False)
                    rotated = False
                    if pos_info is None:
                        pos_info = self._find_stacking_position(cid, rotated=True)
                        rotated = True
                    if pos_info and pos_info[0] == self.target_pos:
                        if rotated:
                            self.containers[cid]['rotated'] = True
                            self.rotation_msg = f"Rotasyon: {self.containers[cid]['name']}"
                        (r, c, level), size = pos_info
                        for i in range(size[0]):
                            for j in range(size[1]):
                                stack = self.stacking_grid[r-3+i][c+j]
                                while len(stack) < level: stack.append(None)
                                stack.append(cid)
                        self.containers[cid]['delivered'] = True
                        self.carrying1 = None
                        delivered = sum(1 for o in self.orders if self.containers[o]['delivered'])
                        reward += 10.0 * delivered
                        self._update_targets()
                        self._prev_distance1 = self._get_distance(self.crane1_pos, self._get_current_target())
                        if all(self.containers[o]['delivered'] for o in self.orders):
                            reward += 20.0
                            self.done = True
                    else: reward -= 4.0; self.rotation_msg = "Rotasyonla bile sigmiyor!"
            else: reward -= 2.0

        if blocked:
            reward -= 0.1

        if self.carrying1 is None and self.next_pickup_target is not None:
            if tuple(self.crane1_pos) == self.next_pickup_target:
                reward += 1.0

        if self.steps >= self.max_steps:
            self.done = True

        return self._get_obs(), reward, self.done, False, {'rotation': self.rotation_msg}

    # ---------- Render ----------
    def render(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(12, 10))
        grid = np.zeros((self.rows, self.cols))
        for cell in self.blocked_cells:
            grid[cell[0], cell[1]] = 1
        for r in range(self.stacking_height):
            for c in range(self.stacking_width):
                stack = self.stacking_grid[r][c]
                if stack:
                    top_id = stack[-1]
                    level = len(stack)-1
                    if level >= 1: grid[3+r][c] = 6
                    else:
                        col = self.containers[top_id]['color']
                        grid[3+r][c] = 3 if col=='green' else (4 if col=='yellow' else 5)
                else: grid[3+r][c] = 2
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                for cell in data['cells']:
                    col = data['color']
                    grid[cell[0]][cell[1]] = 3 if col=='green' else (4 if col=='yellow' else 5)
        colors = ['white', 'gray', 'lightblue', 'green', 'gold', 'red', 'purple']
        cmap = mcolors.ListedColormap(colors)
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=6)
        for i in range(self.rows+1): ax.axhline(i-0.5, color='black', lw=0.5)
        for j in range(self.cols+1): ax.axvline(j-0.5, color='black', lw=0.5)
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                cells = data['cells']
                cr = sum(c[0] for c in cells)/len(cells)
                cc = sum(c[1] for c in cells)/len(cells)
                ax.text(cc, cr, data['name'], ha='center', va='center', fontsize=6, fontweight='bold')
        for r in range(self.stacking_height):
            for c in range(self.stacking_width):
                stack = self.stacking_grid[r][c]
                if stack:
                    top = stack[-1]; lvl = len(stack)-1
                    lbl = f'K{top}' + (f'\nL{lvl}' if lvl>0 else '')
                    ax.text(c, 3+r, lbl, ha='center', va='center', fontsize=6, fontweight='bold',
                            color='white' if lvl>=1 else 'black')
        if self.target_pos is not None:
            tr, tc, tl = self.target_pos
            ax.add_patch(Rectangle((tc-0.5, tr-0.5),1,1, lw=2, edgecolor='yellow', fc='none', ls='--'))
        if self.next_pickup_target is not None and self.carrying1 is None:
            pr, pc = self.next_pickup_target
            ec = 'green' if self.expected_priority==0 else ('yellow' if self.expected_priority==1 else 'red')
            ax.add_patch(Rectangle((pc-0.5, pr-0.5),1,1, lw=2, edgecolor=ec, fc='none', ls='--'))
        ax.plot(self.crane1_pos[1], self.crane1_pos[0], 'r*', markersize=15, markeredgecolor='black', markeredgewidth=2)
        if self.carrying1 is not None:
            ax.text(self.crane1_pos[1], self.crane1_pos[0]-0.4, f'[{self.containers[self.carrying1]["name"]}]',
                    ha='center', va='center', fontsize=7, color='red', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        ax.plot(self.crane2_pos[1], self.crane2_pos[0], 'b*', markersize=15, markeredgecolor='black', markeredgewidth=2)
        if self.carrying2 is not None:
            ax.text(self.crane2_pos[1], self.crane2_pos[0]-0.4, f'[{self.containers[self.carrying2]["name"]}]',
                    ha='center', va='center', fontsize=7, color='blue', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        ax.plot(self.crane2_home[1], self.crane2_home[0], 's', color='cyan', markersize=8, alpha=0.5)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f'Siparis: {len(self.orders)} konteyner | V1: {self.containers[self.carrying1]["name"] if self.carrying1 else "Bos"} | V2: {self.containers[self.carrying2]["name"] if self.carrying2 else "Bos"}\n{self.rotation_msg}', fontsize=10)
        return ax

# ============================================
# DQN AGENT (state_size=64)
# ============================================
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, action_size)
        )
    def forward(self, x): return self.network(x)

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size; self.action_size = action_size
        self.memory = deque(maxlen=20000)
        self.gamma = 0.95; self.epsilon = 1.0; self.epsilon_min = 0.01; self.epsilon_decay = 0.9995
        self.learning_rate = 0.0003; self.batch_size = 128
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DQN(state_size, action_size).to(self.device)
        self.target_model = DQN(state_size, action_size).to(self.device)
        self.update_target_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()
    def update_target_model(self): self.target_model.load_state_dict(self.model.state_dict())
    def remember(self, *args): self.memory.append(args)
    def act(self, state, training=True):
        if training and np.random.rand() <= self.epsilon: return random.randrange(self.action_size)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad(): q = self.model(state_t)
        return q.cpu().numpy()[0].argmax()
    def replay(self):
        if len(self.memory) < self.batch_size: return
        batch = random.sample(self.memory, self.batch_size)
        states = np.array([b[0] for b in batch])
        actions = np.array([b[1] for b in batch])
        rewards = np.array([b[2] for b in batch])
        next_states = np.array([b[3] for b in batch])
        dones = np.array([b[4] for b in batch])
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.LongTensor(actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        curr_q = self.model(states).gather(1, actions.unsqueeze(1))
        with torch.no_grad(): max_next_q = self.target_model(next_states).max(1)[0]
        target_q = rewards + (1-dones) * self.gamma * max_next_q
        loss = self.criterion(curr_q.squeeze(), target_q)
        self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()
        if self.epsilon > self.epsilon_min: self.epsilon *= self.epsilon_decay

# ============================================
# EĞİTİM (ÇİFT AJAN)
# ============================================
print("Ortam oluşturuluyor...")
env = ContainerYardEnv()
state_size = env.state_size
action_size = 6

agent1 = DQNAgent(state_size, action_size)  # Vinç-1
agent2 = DQNAgent(state_size, action_size)  # Vinç-2

episodes = 2000
print(f"Eğitim başlıyor... ({episodes} bölüm)")

rewards_history = []
for e in tqdm(range(episodes), desc="Eğitim"):
    state, _ = env.reset()
    total_reward = 0
    while True:
        action1 = agent1.act(state)
        action2 = agent2.act(state)
        next_state, reward, done, _, _ = env.step(action1, action2)

        agent1.remember(state, action1, reward, next_state, done)
        agent2.remember(state, action2, reward, next_state, done)

        state = next_state
        total_reward += reward

        if done:
            agent1.update_target_model()
            agent2.update_target_model()
            break

        if len(agent1.memory) > agent1.batch_size:
            agent1.replay()
        if len(agent2.memory) > agent2.batch_size:
            agent2.replay()

    rewards_history.append(total_reward)
    if (e+1) % 200 == 0:
        avg = np.mean(rewards_history[-200:])
        tqdm.write(f"Bölüm: {e+1}/{episodes}, Ort. Ödül: {avg:.2f}, E1: {agent1.epsilon:.3f}, E2: {agent2.epsilon:.3f}")

print("Eğitim tamamlandı!")
plt.figure(figsize=(10,4))
plt.plot(rewards_history); plt.xlabel('Bölüm'); plt.ylabel('Ödül'); plt.title('Eğitim Süreci (10x10 Çift Ajan)'); plt.grid(); plt.show()

# ============================================
# TEST VE GIF
# ============================================
print("Test başlıyor...")
test_env = ContainerYardEnv()
state, _ = test_env.reset()
print(f"Siparişler: {', '.join([test_env.containers[o]['name'] for o in test_env.orders])}")
frames = []
fig, ax = plt.subplots(figsize=(12,10))
step = 0
max_steps = 1500
while step < max_steps:
    ax.clear()
    test_env.render(ax)
    fig.canvas.draw()
    img = np.array(fig.canvas.renderer.buffer_rgba())
    frames.append(img[:,:,:3])
    action1 = agent1.act(state, training=False)
    action2 = agent2.act(state, training=False)
    state, reward, done, _, _ = test_env.step(action1, action2)
    step += 1
    if done:
        ax.clear(); test_env.render(ax); fig.canvas.draw()
        img = np.array(fig.canvas.renderer.buffer_rgba()); frames.append(img[:,:,:3])
        break
plt.close()
gif_path = '10x10_multi_agent.gif'
imageio.mimsave(gif_path, frames, fps=5, loop=0)
print(f"Test tamamlandı ({step} adım), GIF: {gif_path}")
from IPython.display import Image as IPImage
IPImage(filename=gif_path)